In [4]:
import pandas as pd
import numpy as np
import re, string, pickle, warnings
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow:", tf.__version__)
print("✅ Import selesai")

TensorFlow: 2.21.0
✅ Import selesai


In [5]:
df = pd.read_csv('dataset.csv')
df.dropna(subset=['review', 'score'], inplace=True)
df.drop_duplicates(subset=['review'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Labeling 3 kelas
def label_sentiment(score):
    if score <= 2:   return 'negatif'
    elif score == 3: return 'netral'
    else:            return 'positif'

df['sentiment'] = df['score'].apply(label_sentiment)

print(f"Total data: {len(df):,}")
print(df['sentiment'].value_counts())
df.head(3)

Total data: 13,768
sentiment
negatif    7793
positif    4880
netral     1095
Name: count, dtype: int64


,app,review,score,date,sentiment
0,Gojek,"karya terbaik bangsa, very helpfull map. Pesan...",5,2026-05-08 09:26:06,positif
1,Gojek,BANYAK PROMONYA🤧,5,2026-05-08 09:19:17,positif
2,Gojek,Driver cepat datangnya dan sampai di tujuan de...,5,2026-05-08 09:18:09,positif


In [ ]:
STOPWORDS = set("""
yang dan di ke dari ini itu ada dengan untuk juga sudah saya kita
tidak bisa lebih aja sih dong lah yg ga gak gk tdk udah udh gue gw
lu deh nya kalau kalo banget blm belum sama mereka kami anda bagi
oleh karena seperti serta atau tapi jadi akan pun sedang
masih banyak hanya mau saja sangat sekali bisa kan nih mana harus
kok oh ya nah pake pakai emang memang pernah ingat biasa coba
""".split())

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'@\w+|#\w+', ' ', text)
    # Hapus semua karakter non-ASCII (mencakup semua emoji & simbol)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    words = [w for w in text.split() if w not in STOPWORDS and len(w) > 1]
    return ' '.join(words)

df['clean_text'] = df['review'].apply(preprocess)
df = df[df['clean_text'].str.strip().str.len() > 3].reset_index(drop=True)

print(f"Data setelah preprocessing: {len(df):,}")
print(df['clean_text'].head(3))

Data setelah preprocessing: 13,686
0    karya terbaik bangsa very helpfull map pesan s...
1                                             promonya
2            driver cepat datangnya sampai tujuan aman
Name: clean_text, dtype: object


: 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#6bcb77', '#ffd93d', '#ff6b6b']

# Pie chart
sc = df['sentiment'].value_counts()
axes[0].pie(sc.values, labels=sc.index, autopct='%1.1f%%', colors=colors)
axes[0].set_title('Distribusi Sentimen')

# Panjang review
df['rev_len'] = df['clean_text'].apply(len)
axes[1].hist(df['rev_len'].clip(upper=300), bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Panjang Review (clean)')

# WordCloud positif
corpus = ' '.join(df[df['sentiment'] == 'positif']['clean_text'])
wc = WordCloud(width=600, height=300, background_color='white', colormap='Greens').generate(corpus)
axes[2].imshow(wc, interpolation='bilinear')
axes[2].axis('off')
axes[2].set_title('WordCloud — Positif')

plt.tight_layout()
plt.savefig('eda.png', dpi=120)
plt.show()
print("✅ EDA selesai")

In [ ]:
le = LabelEncoder()
le.fit(['negatif', 'netral', 'positif'])
df['label'] = le.transform(df['sentiment'])

X = df['clean_text'].values
y = df['label'].values

# Konstanta LSTM
MAX_VOCAB     = 15000
MAX_LEN       = 80
EMBEDDING_DIM = 64

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print(f"Total sampel: {len(X):,}")

In [ ]:
print("=" * 55)
print("SKEMA 1 — BiLSTM + Keras Embedding | Split 80/20")
print("=" * 55)

# Tokenisasi
tok = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tok.fit_on_texts(X)
X_seq = tok.texts_to_sequences(X)
X_pad = pad_sequences(X_seq, maxlen=MAX_LEN, padding='post', truncating='post')

# Split
X_tr1, X_te1, y_tr1, y_te1 = train_test_split(
    X_pad, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(y_tr1):,} | Test: {len(y_te1):,}")

# Model
model1 = Sequential([
    Embedding(MAX_VOCAB, EMBEDDING_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    Dense(32, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax'),
])
model1.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model1.summary()

# Training
history1 = model1.fit(
    X_tr1, y_tr1,
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)],
    verbose=1,
)

# Evaluasi
train_acc1 = model1.evaluate(X_tr1, y_tr1, verbose=0)[1]
test_acc1  = model1.evaluate(X_te1, y_te1, verbose=0)[1]
y_pred1    = np.argmax(model1.predict(X_te1, verbose=0), axis=1)

print(f"\n📊 Skema 1 | Train: {train_acc1*100:.2f}% | Test: {test_acc1*100:.2f}%")
print(classification_report(y_te1, y_pred1, target_names=le.classes_))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_te1, y_pred1), display_labels=le.classes_).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Skema 1 — BiLSTM')
plt.tight_layout(); plt.savefig('cm_skema1.png', dpi=120); plt.show()

# Simpan
model1.save('model_bilstm.h5')
with open('tokenizer_bilstm.pkl', 'wb') as f: pickle.dump(tok, f)
print("✅ Skema 1 tersimpan.")

In [ ]:
print("=" * 55)
print("SKEMA 2 — SVM (LinearSVC) + TF-IDF | Split 80/20")
print("=" * 55)

tfidf2 = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
X_tfidf2 = tfidf2.fit_transform(X)

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_tfidf2, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(y_tr2):,} | Test: {len(y_te2):,} | Fitur: {X_tfidf2.shape[1]:,}")

svm = LinearSVC(C=1.0, max_iter=3000, class_weight='balanced', random_state=42)
svm.fit(X_tr2, y_tr2)

train_acc2 = accuracy_score(y_tr2, svm.predict(X_tr2))
y_pred2    = svm.predict(X_te2)
test_acc2  = accuracy_score(y_te2, y_pred2)

print(f"\n📊 Skema 2 | Train: {train_acc2*100:.2f}% | Test: {test_acc2*100:.2f}%")
print(classification_report(y_te2, y_pred2, target_names=le.classes_))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_te2, y_pred2), display_labels=le.classes_).plot(ax=ax, cmap='Greens', colorbar=False)
ax.set_title('Skema 2 — SVM + TF-IDF')
plt.tight_layout(); plt.savefig('cm_skema2.png', dpi=120); plt.show()

with open('model_svm.pkl', 'wb') as f:  pickle.dump(svm, f)
with open('tfidf_svm.pkl', 'wb') as f:  pickle.dump(tfidf2, f)
print("✅ Skema 2 tersimpan.")

In [ ]:
print("=" * 55)
print("SKEMA 3 — Random Forest + TF-IDF | Split 70/30")
print("=" * 55)

tfidf3 = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), sublinear_tf=True)
X_tfidf3 = tfidf3.fit_transform(X)

X_tr3, X_te3, y_tr3, y_te3 = train_test_split(
    X_tfidf3, y, test_size=0.3, random_state=42, stratify=y)
print(f"Train: {len(y_tr3):,} | Test: {len(y_te3):,}")

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                            random_state=42, n_jobs=-1)
rf.fit(X_tr3, y_tr3)

train_acc3 = accuracy_score(y_tr3, rf.predict(X_tr3))
y_pred3    = rf.predict(X_te3)
test_acc3  = accuracy_score(y_te3, y_pred3)

print(f"\n📊 Skema 3 | Train: {train_acc3*100:.2f}% | Test: {test_acc3*100:.2f}%")
print(classification_report(y_te3, y_pred3, target_names=le.classes_))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_te3, y_pred3), display_labels=le.classes_).plot(ax=ax, cmap='Oranges', colorbar=False)
ax.set_title('Skema 3 — RF + TF-IDF')
plt.tight_layout(); plt.savefig('cm_skema3.png', dpi=120); plt.show()

with open('model_rf.pkl',   'wb') as f: pickle.dump(rf,     f)
with open('tfidf_rf.pkl',   'wb') as f: pickle.dump(tfidf3, f)
print("✅ Skema 3 tersimpan.")

In [ ]:
summary = pd.DataFrame({
    'Skema':      ['Skema 1', 'Skema 2', 'Skema 3'],
    'Algoritma':  ['BiLSTM (Deep Learning)', 'SVM (LinearSVC)', 'Random Forest'],
    'Fitur':      ['Keras Embedding', 'TF-IDF 1-2gram', 'TF-IDF 1-3gram'],
    'Split':      ['80/20', '80/20', '70/30'],
    'Train Acc':  [f'{train_acc1*100:.2f}%', f'{train_acc2*100:.2f}%', f'{train_acc3*100:.2f}%'],
    'Test Acc':   [f'{test_acc1*100:.2f}%',  f'{test_acc2*100:.2f}%',  f'{test_acc3*100:.2f}%'],
})
print(summary.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x  = np.arange(3)
w  = 0.35
b1 = ax.bar(x - w/2, [train_acc1, train_acc2, train_acc3], w, label='Train', color='steelblue')
b2 = ax.bar(x + w/2, [test_acc1,  test_acc2,  test_acc3],  w, label='Test',  color='coral')
for b in [*b1, *b2]:
    ax.annotate(f'{b.get_height()*100:.1f}%',
                xy=(b.get_x() + b.get_width()/2, b.get_height()),
                xytext=(0, 4), textcoords='offset points', ha='center', fontsize=9)
ax.axhline(0.92, color='red',    linestyle='--', linewidth=1.2, label='Target 92%')
ax.axhline(0.85, color='orange', linestyle='--', linewidth=1.2, label='Minimum 85%')
ax.set_xticks(x)
ax.set_xticklabels(['BiLSTM\n(Skema 1)', 'SVM\n(Skema 2)', 'RF\n(Skema 3)'])
ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1.15)
ax.set_title('Perbandingan Akurasi Ketiga Skema')
ax.legend()
plt.tight_layout(); plt.savefig('comparison.png', dpi=120); plt.show()

In [ ]:
# ── Load artifacts ────────────────────────────────────────────────────────────
with open('tokenizer_bilstm.pkl', 'rb') as f: tok_inf  = pickle.load(f)
with open('model_bilstm.h5',     'rb') as f: pass      # already in memory as model1
with open('tfidf_svm.pkl',       'rb') as f: tfidf_s   = pickle.load(f)
with open('model_svm.pkl',       'rb') as f: svm_inf   = pickle.load(f)
with open('tfidf_rf.pkl',        'rb') as f: tfidf_r   = pickle.load(f)
with open('model_rf.pkl',        'rb') as f: rf_inf    = pickle.load(f)

def predict(text):
    clean = preprocess(text)

    # Skema 1 — BiLSTM
    seq = tok_inf.texts_to_sequences([clean])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    p1  = le.inverse_transform(np.argmax(model1.predict(pad, verbose=0), axis=1))[0]

    # Skema 2 — SVM
    p2 = le.inverse_transform(svm_inf.predict(tfidf_s.transform([clean])))[0]

    # Skema 3 — RF
    p3 = le.inverse_transform(rf_inf.predict(tfidf_r.transform([clean])))[0]

    return {'BiLSTM': p1, 'SVM': p2, 'RandomForest': p3}

# ── Test inference ─────────────────────────────────────────────────────────────
test_cases = [
    "Aplikasinya sangat bagus, cepat dan mudah digunakan!",
    "Buruk sekali, sering error dan uang saya terpotong.",
    "Biasa aja sih, tidak terlalu bagus tidak terlalu jelek.",
    "Mantap, sudah 3 tahun pakai belum pernah ada masalah.",
    "Sangat mengecewakan, CS tidak responsif sama sekali.",
]

print(f"{'Teks':<45} | {'BiLSTM':<10} | {'SVM':<10} | {'RF':<10}")
print("─" * 80)
for t in test_cases:
    r = predict(t)
    short = (t[:43] + '..') if len(t) > 45 else t
    print(f"{short:<45} | {r['BiLSTM']:<10} | {r['SVM']:<10} | {r['RandomForest']:<10}")